# Semana 9 — Apache Spark / PySpark

Resolva cada questão na célula correspondente.
Leia o enunciado completo em `ex7_spark.md` antes de começar.

> **Regras:**
> - Nada de `.toPandas()` ou loops Python para transformações
> - Chame `explain()` em pelo menos uma operação por questão
> - Sempre feche a SparkSession no final (`spark.stop()`)

## Setup — imports comuns

In [1]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

print("Python:", sys.executable)
import pyspark; print("PySpark:", pyspark.__version__)


Python: /usr/local/bin/python3.12
PySpark: 3.5.4


---
## Q1 — SparkSession, leitura e primeiras transformações

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Crie a SparkSession com as configs pedidas
spark = SparkSession.builder \
    .appName("Exercicio_Q1") \
    .master("local[*]") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession criada com sucesso! Versão:", spark.version)

# 2. Crie o DataFrame a partir de lista Python com 10 pedidos
dados_pedidos = [
    (1, 150.0, "entregue"), (2, 80.0, "processando"), (3, 200.0, "entregue"),
    (4, 50.0, "cancelado"), (5, 300.0, "entregue"), (6, 120.0, "entregue"),
    (7, 90.0, "processando"), (8, 400.0, "entregue"), (9, 25.0, "entregue"),
    (10, 600.0, "cancelado")
]
colunas = ["id_pedido", "valor", "status"]
df_pedidos = spark.createDataFrame(dados_pedidos, schema=colunas)

# 3. Aplique filter + withColumn + select (em cadeia)
df_transformado = df_pedidos \
    .filter(col("status") == "entregue") \
    .withColumn("valor_com_taxa", col("valor") * 1.1) \
    .select("id_pedido", "valor_com_taxa")

# 4. Chame explain()
print("--- Plano de Execução ---")
df_transformado.explain()

# 5. Chame show()
print("--- Resultado Final ---")
df_transformado.show()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 13:12:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession criada com sucesso! Versão: 3.5.4
--- Plano de Execução ---
== Physical Plan ==
*(1) Project [id_pedido#0L, (valor#1 * 1.1) AS valor_com_taxa#6]
+- *(1) Filter (isnotnull(status#2) AND (status#2 = entregue))
   +- *(1) Scan ExistingRDD[id_pedido#0L,valor#1,status#2]


--- Resultado Final ---


+---------+------------------+
|id_pedido|    valor_com_taxa|
+---------+------------------+
|        1|             165.0|
|        3|220.00000000000003|
|        5|             330.0|
|        6|             132.0|
|        8|440.00000000000006|
|        9|27.500000000000004|
+---------+------------------+



**Resposta à pergunta de reflexão (escreva aqui):**

_Por que o explain() não executou os dados?_

O explain() não executa os dados porque ele é uma operação voltada ao plano de consulta (query plan), não uma Ação. No Apache Spark, as transformações (como filter, select e withColumn) são Lazy (preguiçosas), o que significa que o Spark apenas constrói um grafo de execução (DAG) sem processar os registros reais.

O método explain() apenas imprime as etapas que o otimizador do Spark (Catalyst) planejou realizar — mostrando os planos Lógico e Físico. O processamento real dos dados só ocorre quando uma Ação é chamada, como o show(), count() ou collect(). Portanto, o explain() serve apenas para inspeção da estratégia de otimização, sem tocar nos dados da lista ou do arquivo.

---
## Q2 — Narrow vs Wide: identificando shuffle boundaries

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

spark = SparkSession.builder.appName("Q2_Narrow_vs_Wide").getOrCreate()

# 1. Crie o DataFrame com 50 registros
data = [(i, f"Produto_{i%5}", i * 10.5) for i in range(1, 51)]
df = spark.createDataFrame(data, ["id", "categoria", "preco"])

# 2. Construa o pipeline: filter -> withColumn -> groupBy -> orderBy
df_final = df.filter(col("preco") > 100) \
             .withColumn("preco_com_desconto", col("preco") * 0.9) \
             .groupBy("categoria") \
             .agg(avg("preco_com_desconto").alias("media_preco")) \
             .orderBy("media_preco")

# 3. Chame explain(mode="formatted")
df_final.explain(mode="formatted")

# Para o experimento de remover/mover o orderBy, basta comentar a última linha
# ou movê-la para antes do groupBy para observar a mudança no plano.

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [3]: [id#114L, categoria#115, preco#116]
Arguments: [id#114L, categoria#115, preco#116], MapPartitionsRDD[31] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [3]: [id#114L, categoria#115, preco#116]
Condition : (isnotnull(preco#116) AND (preco#116 > 100.0))

(3) Project
Output [2]: [categoria#115, (preco#116 * 0.9) AS preco_com_desconto#120]
Input [3]: [id#114L, categoria#115, preco#116]

(4) HashAggregate
Input [2]: [categoria#115, preco_com_desconto#120]
Keys [1]: [categoria#115]
Functions [1]: [partial_avg(preco_com_desconto#120)]
Aggregate Attributes [2]: [sum#133, count#134L]
Results [3]: [categoria#115, sum#135, count#136L]

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

spark = SparkSession.builder.appName("Q2_Narrow_vs_Wide").getOrCreate()

# 1. Crie o DataFrame com 50 registros
data = [(i, f"Produto_{i%5}", i * 10.5) for i in range(1, 51)]
df = spark.createDataFrame(data, ["id", "categoria", "preco"])

# 2. Construa o pipeline: filter -> withColumn -> groupBy -> orderBy
df_final = df.filter(col("preco") > 100) \
             .withColumn("preco_com_desconto", col("preco") * 0.9) \
             .groupBy("categoria") \
             .agg(avg("preco_com_desconto").alias("media_preco")) \
            #  .orderBy("media_preco")

# 3. Chame explain(mode="formatted")
df_final.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [3]: [id#137L, categoria#138, preco#139]
Arguments: [id#137L, categoria#138, preco#139], MapPartitionsRDD[36] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [3]: [id#137L, categoria#138, preco#139]
Condition : (isnotnull(preco#139) AND (preco#139 > 100.0))

(3) Project
Output [2]: [categoria#138, (preco#139 * 0.9) AS preco_com_desconto#143]
Input [3]: [id#137L, categoria#138, preco#139]

(4) HashAggregate
Input [2]: [categoria#138, preco_com_desconto#143]
Keys [1]: [categoria#138]
Functions [1]: [partial_avg(preco_com_desconto#143)]
Aggregate Attributes [2]: [sum#156, count#157L]
Results [3]: [categoria#138, sum#158, count#159L]

(5) Exchange
Input [3]: [categoria#138, sum#158, count#159L]
Argu

**Identificação no plano — baseada no output real:**

**Número de stages:**
- Com `orderBy`: **3 stages** (2 shuffles = 2 stage boundaries)
- Sem `orderBy`: **2 stages** (1 shuffle = 1 stage boundary)

**Shuffle boundaries (nós `Exchange`):**
- Exchange (5): `hashpartitioning(categoria, 200)` — antes do `HashAggregate` final. Garante que todos os registros da mesma categoria cheguem ao mesmo executor para a agregação.
- Exchange (7): `rangepartitioning(media_preco, 200)` — antes do `Sort`. Necessário para ordenação **global** entre todos os executors (cada executor precisa ver o intervalo de valores que lhe compete).

**Transformações narrow** (dentro da mesma partição, sem mover dados):
- `Filter` (2), `Project`/`withColumn` (3), `HashAggregate` partial (4)

**Transformações wide** (exigem shuffle):
- `groupBy + agg` → Exchange (5) com `hashpartitioning`
- `orderBy` → Exchange (7) com `rangepartitioning`

**Qual shuffle removeria:** o do `orderBy`. Em ETL, ordenação global é raramente necessária — o consumidor downstream (banco, BI tool, Parquet particionado) tem seus próprios mecanismos de ordenação ou não depende da ordem.

**Detalhe crítico — Two-phase aggregation:**
O Catalyst inseriu `partial_avg` **antes** do shuffle: o nó (4) calcula `sum` e `count` localmente em cada partição e envia apenas esses dois números pelo shuffle. O nó (6) combina com `sum/count` para a média final. Isso pode reduzir em 10–100x o volume de dados transferidos na rede.

**Detalhe de performance — `hashpartitioning(categoria, 200)`:**
200 partições para 5 categorias únicas → 195 partições vazias, 195 tasks que rodam e não fazem nada. Usar `spark.conf.set("spark.sql.shuffle.partitions", "4")` ou ativar AQE (`spark.sql.adaptive.enabled=true`) resolve isso automaticamente.

---
## Q3 — DataFrame API: filter, groupBy, join, window functions

In [ ]:

import random
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Q3").getOrCreate()

# ── df_clientes ────────────────────────────────────────────────────────────────
clientes = [
    (1, "João Silva",     "Premium"),
    (2, "Maria Oliveira", "Standard"),
    (3, "Pedro Costa",    "Premium"),
    (4, "Lucia Alves",    "Standard"),
    (5, "Roberto Nunes",  "Premium"),
]
df_clientes = spark.createDataFrame(clientes, ["id_cliente", "nome", "segmento"])

# ── df_pedidos ─────────────────────────────────────────────────────────────────
# 20 pedidos: clientes 1–5, categorias variadas, valores e meses distribuídos
random.seed(42)
categorias = ["Eletrônicos", "Roupas", "Livros", "Alimentos"]
pedidos = [
    (i, (i % 5) + 1, round(random.uniform(50, 1500), 2),
     categorias[i % 4], (i % 12) + 1)
    for i in range(1, 21)
]
df_pedidos = spark.createDataFrame(
    pedidos, ["id_pedido", "id_cliente", "valor", "categoria", "mes"]
)

print("=== df_clientes ===")
df_clientes.show()
print("=== df_pedidos ===")
df_pedidos.show()


In [ ]:

# Q3.1 — Join + filtro por segmento
# Objetivo: pedidos feitos por clientes Premium, com nome do cliente

df_q31 = (
    df_pedidos
    .join(df_clientes, on="id_cliente", how="inner")   # join pela FK
    .filter(F.col("segmento") == "Premium")            # só clientes Premium
    .select("id_pedido", "nome", "segmento", "categoria", "valor", "mes")
    .orderBy("id_pedido")
)

print("=== Q3.1 — Pedidos de clientes Premium ===")
df_q31.explain(mode="formatted")
df_q31.show(truncate=False)


In [ ]:

# Q3.2 — Ranking por cliente (Window Function)
# Objetivo: ranquear pedidos de cada cliente por valor (maior = rank 1)

# Window particionado por cliente, ordenado por valor desc
janela_cliente = Window.partitionBy("id_cliente").orderBy(F.col("valor").desc())

df_q32 = (
    df_pedidos
    .withColumn("rank_valor", F.rank().over(janela_cliente))
    .join(df_clientes, on="id_cliente", how="inner")
    .select("id_cliente", "nome", "id_pedido", "valor", "rank_valor")
    .orderBy("id_cliente", "rank_valor")
)

print("=== Q3.2 — Ranking de pedidos por cliente ===")
df_q32.explain(mode="formatted")
df_q32.show(truncate=False)

# Diferença entre rank(), dense_rank() e row_number():
# rank()        → pula números em caso de empate (1, 1, 3)
# dense_rank()  → não pula (1, 1, 2)
# row_number()  → único sempre, ordem arbitrária nos empates (1, 2, 3)


In [ ]:

# Q3.3 — Crescimento mês a mês com lag()
# Objetivo: faturamento total por mês e variação percentual em relação ao mês anterior

df_mensal = (
    df_pedidos
    .groupBy("mes")
    .agg(F.round(F.sum("valor"), 2).alias("faturamento"))
    .orderBy("mes")
)

# Window ordenada por mês — sem partitionBy porque queremos sequência global
janela_mes = Window.orderBy("mes")

df_q33 = (
    df_mensal
    .withColumn("fat_mes_anterior", F.lag("faturamento", 1).over(janela_mes))
    .withColumn(
        "crescimento_pct",
        F.round(
            (F.col("faturamento") - F.col("fat_mes_anterior"))
            / F.col("fat_mes_anterior") * 100,
            2
        )
    )
)

print("=== Q3.3 — Crescimento mês a mês ===")
df_q33.explain(mode="formatted")
df_q33.show(truncate=False)

# lag(col, offset, default)
# offset=1 → valor da linha anterior na janela
# Primeira linha retorna null (sem mês anterior) — normal e esperado


In [ ]:

# Q3.4 — Top categoria por valor médio
# Objetivo: categoria com maior ticket médio, usando uma única expressão encadeada

df_q34 = (
    df_pedidos
    .groupBy("categoria")
    .agg(F.round(F.avg("valor"), 2).alias("ticket_medio"))
    .orderBy(F.col("ticket_medio").desc())
    .limit(1)   # top 1 — equivalent a LIMIT 1 no SQL
)

print("=== Q3.4 — Top categoria por ticket médio ===")
df_q34.explain(mode="formatted")
df_q34.show(truncate=False)

# Alternativa com Window (sem limit) — ranqueia todas as categorias:
janela_rank = Window.orderBy(F.col("ticket_medio").desc())
df_todas_categorias = (
    df_pedidos
    .groupBy("categoria")
    .agg(F.round(F.avg("valor"), 2).alias("ticket_medio"))
    .withColumn("rank", F.dense_rank().over(janela_rank))
)
print("=== Ranking completo de categorias ===")
df_todas_categorias.show(truncate=False)


---
## Q4 — Performance: broadcast join, shuffle.partitions, AQE

In [ ]:

# Q4.1 — Crie df_grande (1000 registros) e df_categorias (10 registros)

import random
random.seed(7)

# df_grande: tabela de fatos — simula transações
dados_grandes = [
    (i, f"prod_{i}", round(random.uniform(10, 5000), 2), f"cat_{(i % 10) + 1}")
    for i in range(1, 1001)
]
df_grande = spark.createDataFrame(
    dados_grandes, ["id_transacao", "produto", "valor", "id_categoria"]
)

# df_categorias: tabela de dimensão — pequena, candidata a broadcast
dados_cat = [
    (f"cat_{i}", f"Categoria {i}", ["A", "B", "C"][i % 3])
    for i in range(1, 11)
]
df_categorias = spark.createDataFrame(
    dados_cat, ["id_categoria", "nome_categoria", "grupo"]
)

print(f"df_grande: {df_grande.count()} linhas")
df_grande.show(5)
print(f"df_categorias: {df_categorias.count()} linhas")
df_categorias.show()


In [ ]:

# Q4.2 — Cenário A: join padrão (broadcast desativado)
# Desativamos o broadcast automático do Spark para forçar SortMergeJoin ou ShuffledHashJoin

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # -1 = desativa

df_join_a = df_grande.join(df_categorias, on="id_categoria", how="inner")

print("=== Cenário A — Join SEM broadcast ===")
df_join_a.explain(mode="formatted")
# Procure por: SortMergeJoin ou ShuffledHashJoin → ambas os lados fazem Exchange (shuffle)


In [ ]:

# Q4.3 — Cenário B: broadcast join explícito
# broadcast() envolve o df pequeno — o Spark copia df_categorias para CADA executor
# sem nenhum shuffle em nenhum dos lados

from pyspark.sql.functions import broadcast

df_join_b = df_grande.join(broadcast(df_categorias), on="id_categoria", how="inner")

print("=== Cenário B — Join COM broadcast ===")
df_join_b.explain(mode="formatted")
# Procure por: BroadcastHashJoin → apenas o lado grande é lido localmente
# Não há Exchange (shuffle) para df_categorias


In [ ]:

# Q4.4 — Experimento com shuffle.partitions: 200 vs 4

# ── Cenário 200 (padrão do Spark) ─────────────────────────────────────────────
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # broadcast desativado

df_agg_200 = (
    df_grande
    .groupBy("id_categoria")
    .agg(F.sum("valor").alias("total"))
)
print("=== shuffle.partitions = 200 ===")
df_agg_200.explain(mode="formatted")
# O Exchange mostrará: hashpartitioning(id_categoria, 200)
# 200 tarefas no Spark UI — a maioria processando 0 registros (10 categorias únicas)

# ── Cenário 4 (ajustado ao volume real) ───────────────────────────────────────
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_agg_4 = (
    df_grande
    .groupBy("id_categoria")
    .agg(F.sum("valor").alias("total"))
)
print("=== shuffle.partitions = 4 ===")
df_agg_4.explain(mode="formatted")
# O Exchange mostrará: hashpartitioning(id_categoria, 4)
# 4 tarefas no Spark UI — muito mais eficiente para este volume

# Restaura default
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # 10MB



**Resposta (escreva aqui):**

- **Estratégia no Cenário A:** `SortMergeJoin` (ou `ShuffledHashJoin`). Com broadcast desativado, o Spark embaralha (shuffle) **os dois lados** da junção para que registros com a mesma chave cheguem ao mesmo executor. Custo: dois `Exchange` no plano — um para cada DataFrame.

- **Estratégia no Cenário B:** `BroadcastHashJoin`. O Spark serializa `df_categorias` inteiro e envia uma cópia para **cada executor**. Cada executor faz o join localmente contra sua partição de `df_grande`, sem nenhum shuffle. Custo: zero `Exchange` no plano.

- **Por que broadcast é mais rápido:** No SortMergeJoin, ambos os DataFrames precisam ser redistribuídos pela rede (shuffle) — isso envolve serialização, transferência de bytes entre nós, deserialização e sort. No BroadcastHashJoin, apenas o DataFrame pequeno trafega pela rede (uma vez, para cada executor), e o join é resolvido via hash lookup em memória. Para tabelas de dimensão pequenas (< 10 MB, tipicamente), o custo de broadcast é marginal comparado ao shuffle evitado.

- **Risco de broadcast em tabela grande:** Se você fizer `broadcast(df_grande)` com 1 GB de dados, o driver precisará coletar esse DataFrame, serializar e enviar 1 GB para **cada executor**. Isso pode saturar a rede, estourar a memória dos executores (e do driver) e causar OOM (`OutOfMemoryError`). A regra prática é usar broadcast apenas para tabelas que cabem confortavelmente na memória de um executor — o default do Spark é 10 MB (`spark.sql.autoBroadcastJoinThreshold`).

- **Efeito de shuffle.partitions=200 para dados pequenos:** Com 1000 registros e 10 categorias únicas, o Spark cria 200 partições após o shuffle — mas apenas 10 contêm dados. As outras 190 são partições vazias que o scheduler ainda precisa agendar, inicializar e finalizar como tasks. O Spark UI mostrará 200 tasks, a maioria com 0 registros e duração de milissegundos — overhead puro. Reduzir para 4 (número próximo ao de categorias únicas) concentra o trabalho, elimina o overhead e reduz o tempo total. Em produção, use `spark.sql.adaptive.enabled=true` para que o AQE (Adaptive Query Execution) ajuste automaticamente o número de partições pós-shuffle com base no volume real dos dados.


---
## Q5 — Leitura e escrita de Parquet particionado

In [ ]:

# Q5.1 — Crie DataFrame com 200 transações (ano 2022-2024, meses 1-12)

import random
random.seed(99)

anos = [2022, 2023, 2024]
meses = list(range(1, 13))
categorias = ["Eletrônicos", "Roupas", "Livros", "Alimentos", "Esportes"]

transacoes = [
    (
        i,
        random.choice(anos),
        random.choice(meses),
        random.choice(categorias),
        round(random.uniform(20, 3000), 2)
    )
    for i in range(1, 201)
]

df_transacoes = spark.createDataFrame(
    transacoes, ["id_transacao", "ano", "mes", "categoria", "valor"]
)

print(f"Total de transações: {df_transacoes.count()}")
df_transacoes.show(10)
df_transacoes.printSchema()


In [ ]:

# Q5.2 — Escreva particionado por ano e mes

output_path = "output/transacoes"

(
    df_transacoes
    .write
    .mode("overwrite")
    .partitionBy("ano", "mes")          # cria hierarquia de diretórios ano=X/mes=Y/
    .parquet(output_path)
)

print(f"Escrito em: {output_path}")

# Explora a estrutura de diretórios gerada
import os

print("\n=== Estrutura de diretórios ===")
for raiz, dirs, arquivos in os.walk(output_path):
    nivel = raiz.replace(output_path, "").count(os.sep)
    indent = "  " * nivel
    print(f"{indent}{os.path.basename(raiz)}/")
    if nivel == 2:   # só mostra arquivos nas partições folha (ano=X/mes=Y/)
        for arq in arquivos[:1]:
            print(f"{indent}  {arq}")
        if len(arquivos) > 1:
            print(f"{indent}  ... ({len(arquivos)} arquivo(s))")


In [ ]:

# Q5.3 — Leia com filtro e chame explain() — procure por PartitionFilters

df_leitura = spark.read.parquet("output/transacoes")

# Filtra só 2023 — o Spark deve usar Partition Pruning: ler apenas ano=2023/
df_filtrado = df_leitura.filter(F.col("ano") == 2023)

print("=== explain() com filtro de partição ===")
df_filtrado.explain(mode="formatted")

# O que procurar no plano:
# PartitionFilters: [isnotnull(ano#X), (ano#X = 2023)]
#   → Spark NÃO leu as partições ano=2022 e ano=2024
# PushedFilters vs PartitionFilters:
#   - PartitionFilters → eliminação de diretórios inteiros (Partition Pruning)
#   - PushedFilters    → filtro dentro do arquivo Parquet (Predicate Pushdown)

print(f"\nRegistros em 2023: {df_filtrado.count()}")

# Compare: filtrar por coluna NÃO-particionada (categoria) não faz Partition Pruning
df_sem_pruning = df_leitura.filter(F.col("categoria") == "Livros")
print("\n=== explain() SEM PartitionFilters (filtro por categoria) ===")
df_sem_pruning.explain(mode="formatted")
# Repare: nenhum PartitionFilters — o Spark lê TODOS os diretórios


In [ ]:

# Q5.4 — Leia diretamente uma partição específica

df_jun_2023 = spark.read.parquet("output/transacoes/ano=2023/mes=6/")

print("=== Partição ano=2023/mes=6 ===")
df_jun_2023.show(truncate=False)
print(f"Registros: {df_jun_2023.count()}")

# Atenção: ao ler uma partição diretamente, as colunas de partição (ano, mes)
# NÃO aparecem automaticamente no schema — o Spark as infere do path apenas
# quando você lê o diretório raiz com spark.read.parquet("output/transacoes")
print("\nSchema ao ler partição diretamente (sem ano/mes):")
df_jun_2023.printSchema()

print("\nSchema ao ler raiz (com ano e mes como colunas):")
spark.read.parquet("output/transacoes").filter(
    (F.col("ano") == 2023) & (F.col("mes") == 6)
).printSchema()



**Resposta (escreva aqui):**

- **Estrutura de diretórios encontrada:**
  ```
  output/transacoes/
    ano=2022/
      mes=1/   → part-00000-....parquet
      mes=2/   → part-00000-....parquet
      ...
      mes=12/  → part-00000-....parquet
    ano=2023/
      mes=1/ ... mes=12/
    ano=2024/
      mes=1/ ... mes=12/
  ```
  Cada combinação `ano=X/mes=Y/` é uma partição física — um diretório contendo um ou mais arquivos Parquet. O Spark usa o nome do diretório (`ano=2023`) para inferir o valor das colunas de partição sem precisar abrir o arquivo.

- **O explain() mostrou PartitionFilters? O que isso significa:**
  Sim. `PartitionFilters: [isnotnull(ano#X), (ano#X = 2023)]` significa que o Spark aplicou **Partition Pruning** — em vez de abrir todos os diretórios de todas as partições, ele identificou no metadado do filesystem quais diretórios satisfazem o filtro `ano == 2023` e leu **apenas** `ano=2023/mes=1/` até `ano=2023/mes=12/`. As partições de 2022 e 2024 nunca foram abertas, nem listadas. Isso é a vantagem central do dado particionado: a seletividade acontece no nível de I/O, antes de qualquer processamento.

- **`(ano, mes, categoria)` vs `(ano, mes)` — qual é melhor para leitura de "todos os dados de 2023":**
  `(ano, mes)` é melhor para essa consulta. Com `(ano, mes, categoria)`, o Spark precisaria de filtros exatos em `ano`, `mes` **e** `categoria` para atingir Partition Pruning eficiente. Um filtro só em `ano=2023` ainda percorreria os 5 subdiretórios de categoria em cada um dos 12 meses (60 diretórios vs 12). Além disso, quanto mais granular a partição, mais arquivos pequenos são criados — o famoso "small files problem", que degrada tanto o Spark (mais tasks) quanto o filesystem (mais metadados). A regra geral: **particione pelas colunas que aparecem nos filtros mais frequentes das queries, do mais geral para o mais específico** — e evite colunas de alta cardinalidade (como `produto_id`) que explodem o número de partições.


---
## Finalização

In [ ]:
spark.stop()